In [1]:
pip install pandas numpy pdfplumber openpyxl

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 1.9 MB/s eta 0:00:06
   --- ------------------------------------ 0.8/9.9 MB 1.2 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/9.9 MB 1.2 MB/s eta 0:00:08
   ---- ----------------------------------- 1.0/9.9 MB 1.0 MB/s eta 0:00:09
   ---- ----------------------------------- 1.0/9.9 MB 1.0 MB/s eta 0:00:09
   ----- ---------------------------------- 1.3/9.9 MB 818.6 kB/s eta 0:00:11
   ----- ---------------------------------- 1.3/9.9 MB 818.6 kB/s eta 0:00:11
   ------ --------------------------------- 1.6/9.9 MB 762.5 kB/s eta 0:00:11
   ------ --------------------------------- 1.6/9.9 MB 762.5 kB/s eta 0:00:11
   ------- -------------------------------- 1.8/9.9 MB 740.2 kB/s eta 0:00:11
   ------- -------------------------------- 1.8/9.9 MB 740.2 kB/s eta 0:00:11
   -------- --

In [4]:
import pandas as pd
import numpy as np

# File paths
wb_file = r"data\all.xlsx"
mospi_pdf1 = r"data\30.pdf"
mospi_pdf2 = r"data\QPISR_4th_QTR_2024-25.pdf"

# Load World Bank Excel
wb_df = pd.read_excel(wb_file, sheet_name=0, header=1)

print(wb_df.columns.tolist())
wb_df.head()

c:\Users\YT\anaconda3\envs\corporate-pm\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['Project ID', 'Region', 'Country', 'Project Status', 'Last Stage Reached Name', 'Project Name', 'Project Development Objective ', 'Implementing Agency', 'Public Disclosure Date', 'Board Approval Date', 'Loan Effective Date', 'Project Closing Date', 'Current Project Cost', 'IBRD Commitment', 'IDA Commitment', 'Grant Amount', 'Total IBRD, IDA and Grant Commitment', 'Borrower', 'Lending Instrument', 'Environmental Assessment Category', 'Environmental and Social Risk', 'Associated Project', 'Consultant Services Required', 'Last Update Date', 'Financing Type']


,Project ID,Region,Country,Project Status,Last Stage Reached Name,Project Name,Project Development Objective,Implementing Agency,Public Disclosure Date,Board Approval Date,...,Grant Amount,"Total IBRD, IDA and Grant Commitment",Borrower,Lending Instrument,Environmental Assessment Category,Environmental and Social Risk,Associated Project,Consultant Services Required,Last Update Date,Financing Type
0,id,regionname,countryshortname,status,last_stage_reached_name,project_name,pdo,impagency,public_disclosure_date,boardapprovaldate,...,grantamt,curr_total_commitment,borrower,lendinginstr,envassesmentcategorycode,esrc_ovrl_risk_rate,supplementprojectflg,cons_serv_reqd_ind,proj_last_upd_date,projectfinancialtype
1,P000001,Africa,Africa,Closed,Bank Approved,West Africa Pilot Community-based Natural Reso...,CONSERVATION OF BIO-DIVERSITY. OBJECTIVE IS T...,NaN,1995-09-14,1995-09-14T00:00:00Z,...,11400000,11400000,NaN,Specific Investment Loan,B,Not Applicable,N,NaN,2013-01-15,Other
2,P000002,Africa,Africa,Dropped,Bank Approved,LAKE VICTORIA ENVIRO,NaN,NaN,NaN,1997-06-30T00:00:00Z,...,NaN,NaN,NaN,NaN,B,NaN,NaN,NaN,NaN,NaN
3,P000003,Africa,Africa,Closed,Bank Approved,REIMP(CEN.ENV.INFO),REIMP TO IMPROVE/STRENGTHEN PLANNING & MNGT. O...,NaN,1997-12-18,1997-12-18T00:00:00Z,...,16700000,16700000,NaN,Specific Investment Loan,C,Not Applicable,N,NaN,2013-01-15,Other
4,P000004,Africa,Africa,Dropped,Bank Approved,P.TA/SADCC TRADE DEV,NaN,NaN,NaN,1991-06-30T00:00:00Z,...,NaN,NaN,NaN,Specific Investment Loan,C,Not Applicable,NaN,NaN,NaN,NaN


In [7]:
for col in wb_df.columns:
    print(f"'{col}'")

'Project ID'
'Region'
'Country'
'Project Status'
'Last Stage Reached Name'
'Project Name'
'Project Development Objective '
'Implementing Agency'
'Public Disclosure Date'
'Board Approval Date'
'Loan Effective Date'
'Project Closing Date'
'Current Project Cost'
'IBRD Commitment'
'IDA Commitment'
'Grant Amount'
'Total IBRD, IDA and Grant Commitment'
'Borrower'
'Lending Instrument'
'Environmental Assessment Category'
'Environmental and Social Risk'
'Associated Project'
'Consultant Services Required'
'Last Update Date'
'Financing Type'


In [8]:
wb_df.columns = (
    wb_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [10]:
# Convert dates
wb_df['board_approval_date'] = pd.to_datetime(wb_df['board_approval_date'], errors='coerce')
wb_df['project_closing_date'] = pd.to_datetime(wb_df['project_closing_date'], errors='coerce')

# Drop missing
# Convert both to datetime and REMOVE timezone
wb_df['board_approval_date'] = pd.to_datetime(
    wb_df['board_approval_date'], errors='coerce'
).dt.tz_localize(None)

wb_df['project_closing_date'] = pd.to_datetime(
    wb_df['project_closing_date'], errors='coerce'
).dt.tz_localize(None)
# Convert cost
wb_df['current_project_cost'] = pd.to_numeric(wb_df['current_project_cost'], errors='coerce')

# Log transform
wb_df['log_project_cost'] = np.log1p(wb_df['current_project_cost'])

# Risk mapping
risk_map = {'Low':1, 'Moderate':2, 'Substantial':3, 'High':4}
wb_df['risk_rating'] = wb_df['environmental_and_social_risk'].map(risk_map)

wb_df.head()

,project_id,region,country,project_status,last_stage_reached_name,project_name,project_development_objective,implementing_agency,public_disclosure_date,board_approval_date,...,borrower,lending_instrument,environmental_assessment_category,environmental_and_social_risk,associated_project,consultant_services_required,last_update_date,financing_type,log_project_cost,risk_rating
1,P000001,Africa,Africa,Closed,Bank Approved,West Africa Pilot Community-based Natural Reso...,CONSERVATION OF BIO-DIVERSITY. OBJECTIVE IS T...,NaN,1995-09-14,1995-09-14,...,NaN,Specific Investment Loan,B,Not Applicable,N,NaN,2013-01-15,Other,16.394970,NaN
3,P000003,Africa,Africa,Closed,Bank Approved,REIMP(CEN.ENV.INFO),REIMP TO IMPROVE/STRENGTHEN PLANNING & MNGT. O...,NaN,1997-12-18,1997-12-18,...,NaN,Specific Investment Loan,C,Not Applicable,N,NaN,2013-01-15,Other,16.630919,NaN
17,P000017,Africa,Africa,Closed,Bank Approved,3A-TG/BN Engineering TAL (FY92),I) INSTITUTIONAL STRENGTHENING; II) A LEAST-CO...,NaN,1992-05-19,1992-05-19,...,NaN,Technical Assistance Loan,C,Not Applicable,N,NaN,2013-01-15,IDA,15.623799,NaN
34,P000034,Eastern and Southern Africa,Angola,Closed,Bank Approved,INFRASTRUCTURE REHAB,"Feasibility and engineering studies, technical...",NaN,1991-07-16,1991-07-16,...,NaN,Technical Assistance Loan,C,Not Applicable,N,NaN,2021-03-11,IDA,16.959663,NaN
35,P000035,Eastern and Southern Africa,Angola,Closed,Bank Approved,LOBITO/BENG.REHAB.,"PROJECT WOULD ASSIST GOVT TO I) RESTORE WATER,...",NaN,1992-01-07,1992-01-07,...,NaN,Sector Investment and Maintenance Loan,C,Not Applicable,N,NaN,2013-01-15,IDA,18.002130,NaN


In [11]:
# Drop missing
wb_df = wb_df.dropna(subset=['board_approval_date', 'project_closing_date'])

# Create duration
wb_df['actual_duration_days'] = (
    wb_df['project_closing_date'] - wb_df['board_approval_date']
).dt.days

In [12]:
import pdfplumber
import pandas as pd

def extract_tables(pdf_path):
    all_tables = []
    
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            
            for table in tables:
                df = pd.DataFrame(table)
                all_tables.append(df)
    
    return all_tables

# Extract both PDFs
tables_1 = extract_tables(mospi_pdf1)
tables_2 = extract_tables(mospi_pdf2)

print("Tables in PDF1:", len(tables_1))
print("Tables in PDF2:", len(tables_2))

Tables in PDF1: 328
Tables in PDF2: 280


In [13]:
tables_1[0].head()

,0
0,484\nFEBRUARY 2026\nScan the QR code to access...
1,"Project Assessment, Infrastructure Monitoring ..."


In [14]:
mospi_df = tables_1[0].copy()

# Set first row as header
mospi_df.columns = mospi_df.iloc[0]
mospi_df = mospi_df[1:]

mospi_df.head()

,484\nFEBRUARY 2026\nScan the QR code to access\nthe PAIMANA Portal.
1,"Project Assessment, Infrastructure Monitoring ..."


In [15]:
mospi_df.columns = mospi_df.columns.str.strip().str.lower().str.replace(" ", "_")

In [16]:
print(mospi_df.columns.tolist())

['484\nfebruary_2026\nscan_the_qr_code_to_access\nthe_paimana_portal.']


In [17]:
for i, table in enumerate(tables_2):
    print(f"\nTable {i}")
    print(pd.DataFrame(table).head())


Table 0
  0         1
0    SYNOPSIS

Table 1
  0
0  
1  
2  

Table 2
  0               1
0    LIST OF TABLES

Table 3
         0                               1         2  \
0  Sl. No.                          Sector  Projects   
1        1                  CIVIL AVIATION        41   
2        2                            COAL       121   
3        3  DEPARTMENT OF HIGHER EDUCATION        20   
4        4                           DONER         1   

                                                   3  \
0  Cost\nOriginal\n(Revised)*\n{Anticipated}\nin ...   
1                28,200.17\n(29,343.52)\n{32,097.25}   
2             208,420.76\n(212,921.45)\n{218,842.29}   
3                 9,680.92\n(10,072.78)\n{10,127.07}   
4                         151.33\n(151.33)\n{151.33}   

                                       4  
0  Cumulative\nExpenditure\nin Rs. Crore  
1                              18,637.74  
2                              76,381.85  
3                               7,